## IMPLEMENTING A GPT MODEL FROM SCRATCH TO GENERATE TEXT


In [1]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,  # Vocabulary size
    "context_length": 1024,  # Context length
    "emb_dim": 768,  # Embedding dimension
    "n_heads": 12,  # Number of attention heads
    "n_layers": 12,  # Number of layers
    "drop_rate": 0.1,  # Dropout rate
    "qkv_bias": False,  # Query-Key-Value bias
}

In [2]:
import torch
import torch.nn as nn


class DummyGPTModel(nn.Module):
    # basic GPT model structure banako
    # transformer ko overall pipeline simulate garxa

    def __init__(self, cfg):
        super().__init__()

        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        # token embedding layer
        # token ids lai dense vectors ma convert garxa

        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        # positional embedding layer
        # token ko position information add garxa

        self.drop_emb = nn.Dropout(cfg["drop_rate"])
        # embedding pachi dropout apply garna use hunxa

        self.trf_blocks = nn.Sequential(
            *[DummyTransformerBlock(cfg) for _ in range(cfg["n_layers"])]
        )
        # multiple transformer blocks stack gareko
        # ahile dummy block use gareko cha

        self.final_norm = DummyLayerNorm(cfg["emb_dim"])
        # final normalization layer
        # ahile placeholder matra ho

        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)
        # final linear layer
        # embedding vectors lai vocabulary logits ma convert garxa

    def forward(self, in_idx):

        batch_size, seq_len = in_idx.shape
        # input ko batch size ra sequence length nikaleko

        tok_embeds = self.tok_emb(in_idx)
        # token embeddings generate gareko

        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        # positional embeddings generate gareko

        x = tok_embeds + pos_embeds
        # token ra positional embeddings add gareko

        x = self.drop_emb(x)
        # embeddings ma dropout apply gareko

        x = self.trf_blocks(x)
        # transformer blocks bata pass gareko

        x = self.final_norm(x)
        # final normalization apply gareko

        logits = self.out_head(x)
        # final output logits generate gareko

        return logits
        # vocabulary prediction return gareko


class DummyTransformerBlock(nn.Module):
    # placeholder transformer block

    def __init__(self, cfg):
        super().__init__()
        # simple dummy initialization

    def forward(self, x):
        # yo block le kei process gardaina
        # input jastai return garxa

        return x


class DummyLayerNorm(nn.Module):
    # placeholder layer normalization

    def __init__(self, normalized_shape, eps=1e-5):
        super().__init__()
        # layer norm ko interface mimic gareko

    def forward(self, x):
        # yo layer le normalization gardaina
        # input direct return garxa

        return x

In [3]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")
# GPT-2 tokenizer load gareko

batch = []
# tokenized sequences store garna empty list banako

txt1 = "Every effort moves you"
txt2 = "Every day holds a"
# sample text inputs define gareko

batch.append(torch.tensor(tokenizer.encode(txt1)))
# first text tokenize garera tensor ma convert gareko

batch.append(torch.tensor(tokenizer.encode(txt2)))
# second text tokenize garera tensor ma convert gareko

batch = torch.stack(batch, dim=0)
# dui ota token tensors lai batch dimension ma combine gareko

print(batch)
# final tokenized batch print gareko
# shape: (batch_size, seq_len)

tensor([[6109, 3626, 6100,  345],
        [6109, 1110, 6622,  257]])


In [4]:
torch.manual_seed(123)
# random initialization same auna ko lagi seed set gareko

model = DummyGPTModel(GPT_CONFIG_124M)
# dummy GPT model create gareko using configuration

logits = model(batch)
# tokenized batch lai model ma pass gareko
# forward pass garera output logits generate gareko

print("Output shape:", logits.shape)
# output tensor ko shape print gareko
# shape: (batch_size, seq_len, vocab_size)

print(logits)
# final logits print gareko
# each token ko vocabulary prediction scores dekhaunxa

Output shape: torch.Size([2, 4, 50257])
tensor([[[-1.2034,  0.3201, -0.7130,  ..., -1.5548, -0.2390, -0.4667],
         [-0.1192,  0.4539, -0.4432,  ...,  0.2392,  1.3469,  1.2430],
         [ 0.5307,  1.6720, -0.4695,  ...,  1.1966,  0.0111,  0.5835],
         [ 0.0139,  1.6754, -0.3388,  ...,  1.1586, -0.0435, -1.0400]],

        [[-1.0908,  0.1798, -0.9484,  ..., -1.6047,  0.2439, -0.4530],
         [-0.7860,  0.5581, -0.0610,  ...,  0.4835, -0.0077,  1.6621],
         [ 0.3567,  1.2698, -0.6398,  ..., -0.0162, -0.1296,  0.3717],
         [-0.2407, -0.7349, -0.5102,  ...,  2.0057, -0.3694,  0.1814]]],
       grad_fn=<UnsafeViewBackward0>)


### Layer Normalization


In [ ]:
torch.manual_seed(123)

batch_example = torch.randn(2, 5)  # A
# random input batch banako
# shape: (2 samples, 5 features)

layer = nn.Sequential(nn.Linear(5, 6), nn.ReLU())
# sequential neural network layer banako
# Linear: 5 features -> 6 features
# ReLU: negative values lai 0 banauxa

out = layer(batch_example)
# input batch lai layer ma pass gareko

print(out)
# final output print gareko
# shape: (2, 6) hunxa

In [ ]:
mean = out.mean(dim=-1, keepdim=True)
# last dimension ko mean calculate gareko
# keepdim=True le output ko dimension preserve garxa

var = out.var(dim=-1, keepdim=True)
# last dimension ko variance calculate gareko
# variance le values kati spread cha bhanera dekhaunxa

print("Mean:\n", mean)
# each sample ko mean print gareko

print("Variance:\n", var)
# each sample ko variance print gareko

In [ ]:
out_norm = (out - mean) / torch.sqrt(var)
# output lai normalize gareko
# mean subtract garera variance ko sqrt le divide gareko

mean = out_norm.mean(dim=-1, keepdim=True)
# normalized output ko mean calculate gareko

var = out_norm.var(dim=-1, keepdim=True)
# normalized output ko variance calculate gareko

print("Normalized layer outputs:\n", out_norm)
# normalized values print gareko

print("Mean:\n", mean)
# normalized output ko mean print gareko
# mean almost 0 hunxa

print("Variance:\n", var)
# normalized output ko variance print gareko
# variance almost 1 hunxa

In [ ]:
torch.set_printoptions(sci_mode=False)
# scientific notation off gareko
# numbers normal decimal format ma print hunxa

print("Mean:\n", mean)
# mean values print gareko

print("Variance:\n", var)
# variance values print gareko

In [ ]:
class LayerNorm(nn.Module):
    # custom layer normalization implementation banako

    def __init__(self, emb_dim):
        super().__init__()

        self.eps = 1e-5
        # small value add gareko
        # division by zero avoid garna use hunxa

        self.scale = nn.Parameter(torch.ones(emb_dim))
        # learnable scaling parameter (gamma)
        # normalized values lai scale garna use hunxa

        self.shift = nn.Parameter(torch.zeros(emb_dim))
        # learnable shift parameter (beta)
        # normalized values lai shift garna use hunxa

    def forward(self, x):

        mean = x.mean(dim=-1, keepdim=True)
        # last dimension ko mean calculate gareko

        var = x.var(dim=-1, keepdim=True, unbiased=False)
        # last dimension ko variance calculate gareko
        # unbiased=False le population variance use garxa

        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        # input normalize gareko
        # mean subtract garera std deviation le divide gareko

        return self.scale * norm_x + self.shift
        # normalized output lai scale ra shift apply gareko

In [ ]:
ln = LayerNorm(emb_dim=5)
# custom layer normalization layer create gareko
# embedding dimension = 5 set gareko

out_ln = ln(batch_example)
# input batch lai layer normalization ma pass gareko

mean = out_ln.mean(dim=-1, keepdim=True)
# normalized output ko mean calculate gareko

var = out_ln.var(dim=-1, unbiased=False, keepdim=True)
# normalized output ko variance calculate gareko

print("Mean:\n", mean)
# normalized output ko mean print gareko
# values almost 0 hunxa

print("Variance:\n", var)
# normalized output ko variance print gareko
# values almost 1 hunxa